# Introduktion til Markov Jump processer og coalescent modellen


## 1. Introduktion

Formålet med denne notebook er at introducere Markov-modeller og vise, hvordan de kan bruges til at analysere genetiske processer bagud i tid.

Jeg starter med en simpel Markov jump proces og analyserer dens egenskaber. Derefter udvider jeg modellen til coalescent-teori, som er central i populationsgenetik.

Notebooken fungerer som fundament for senere modeller som:
- migration (two-island model)
- IM-model
- analyse af baboon-data


## 2. Simpel Markov model

En Markov jump proces bevæger sig mellem forskellige states over tid. Hver state beskrives som en vektor af heltal, der repræsenterer systemets egenskaber.

Overgange mellem states bestemmes af en funktion, som for en given state returnerer:

- hvilke nye states der kan nås  
- med hvilke rater  

Denne funktion kaldes en callback-funktion og er central i opbygningen af modellen.

Denne model svarer til en eksponentiel fordeling, hvor processen kun har et trin før den afsluttes.

Denne model svarer til en eksponentiel fordeling, hvor processen kun har et trin før den afsluttes.

In [ ]:
import phasic 
from phasic import Graph
import numpy as np

def model(state):
    transitions = []
    
    if state[0] < 2:
        child = state.copy()
        child[0] += 1
        transitions.append([child, 2.0])
        
    return transitions

graph = Graph(model, ipv=[1])
graph.plot()

## 3. Analyse af modellen

Jeg kan beregne centrale størrelser som forventning og varians. 

In [ ]:
graph.expectation(), graph.variance()

Den forventede ventetid er 0.5, hvilket svarer til en eksponentiel fordeling med rate 2.

In [ ]:
graph.moments(10)

## 4. Sandsynlighedsfordeling

Jeg kan både beregne den eksakte fordeling og simulere data fra modellen.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
vals = np.linspace(0, 3, 100)

axes[0].plot(vals, graph.pdf(vals))
axes[0].set_title("PDF")

axes[1].plot(vals, graph.cdf(vals))
axes[1].set_title("CDF")

plt.show()

- PDF beskriver tætheden af ventetider  
- CDF beskriver sandsynligheden for at processen er afsluttet før tid $t$

In [ ]:
samples = graph.sample(10000)
samples[:10]

np.mean(samples).item(), graph.expectation()

Her sammenlignes den simulerede middelværdi med den teoretiske forventning.

## 5. Udvidelse til højere dimensioner

Denne model viser, hvordan state space vokser, når man går til flere dimensioner.

In [ ]:
def mesh(state, max_val=None):
    transitions = []
    for i in range(state.size):
        if state[i] <= max_val:
            child = state.copy()
            child[i] += 1
            trans = [child, state.sum()]
            transitions.append(trans)
    return transitions

graph = Graph(mesh, ipv=[1, 1], max_val=2)
graph.plot()

In [ ]:
graph = Graph(mesh, ipv=[1, 1], max_val=8)
print(graph.expectation())
graph.plot()

## 7. Fra Markov model til Coalescent
I de tidligere afsnit havde jeg kun meget simple transitions.

I populationsgenetik er man interesserede i mange lineager, som kan coalescere bagud i tid.

Coalescent modellen beskriver, hvordan individer kan spores tilbage til en fælles forfader (MRCA).

## 8. Coalescent-modellen

State i Coalescent-modellen kan beskrives som en vektor:

(4, 0, 0, 0)

Det betyder:

- 4 singleton-linjer  
- 0 doubletons  
- osv.

Processen starter her og ender i:

(0, 0, 0, 1)

Når to linjer coalescerer:

- antallet af linjer falder med 1  
- en ny gruppe dannes  

Over tid dannes et coalescent tree.


## 9. Konstruktion af state space

In [ ]:
def coalescent_graph(nr_samples):
    state_vector_length = nr_samples
    graph = Graph(state_vector_length)

    initial_state = np.zeros(state_vector_length, dtype=int)
    initial_state[0] = nr_samples

    vertex = graph.find_or_create_vertex(initial_state)
    graph.starting_vertex().add_edge(vertex, 1)

    index = 1
    while index < graph.vertices_length():
        vertex = graph.vertex_at(index)
        state = vertex.state()

        for i in range(nr_samples):
            for j in range(i, nr_samples):
                same = int(i == j)
                if same and state[i] < 2:
                    continue
                if not same and (state[i] < 1 or state[j] < 1):
                    continue
                new_state = state.copy()
                new_state[i] -= 1
                new_state[j] -= 1
                new_state[i+j+1] += 1
                new_vertex = graph.find_or_create_vertex(new_state)
                rate = state[i]*(state[j]-same)/(1+same)
                vertex.add_edge(new_vertex, rate)
        index += 1
    return graph

In [ ]:
graph = coalescent_graph(4)
graph.plot()

In [ ]:
graph.states()

Her kan man se alle states i modellen. 
Hver række repræsenterer en mulig konfiguration af lineager.

In [ ]:
graph.plot()

## 10. Sammenligning af metoder

Der findes to måder at opbygge modellen på:

**Callback-metoden**

- kortere  
- fleksibel  
- lettere at vedligeholde  

**State-space konstruktion**

- mere eksplicit  
- nyttig til forståelse  

## 11. State properties og indexing

I mere komplekse modeller kan state space blive meget stort.

Så her introduceres StateIndexer, som gør det muligt at arbejde systematisk med multidimensionelle states.

In [ ]:
from phasic import Graph, with_ipv, StateIndexer, PropertySet, Property
import numpy as np
import seaborn as sns

### Properties

En Property repræsenterer en dimension i state space.

In [ ]:
descendants = Property('descendants', max_value=10)
descendants

### Single PropertySet

In [ ]:
indexer = StateIndexer(
    lineage=[Property('descendants', max_value=10)]
)
print(f"Total states: {indexer.state_length}")
print(f"PropertySet: {indexer.lineage}")

In [ ]:
# Index to properties (returns dataclass by default)
props = indexer.lineage.i2p(5)
print(f"Index 5: {props}")
print(f"Access via attribute: props.descendants = {props.descendants}")

# Can also get as dict
props_dict = indexer.lineage.i2p(5, as_dict=True)
print(f"As dict: {props_dict}")

# Properties to index
idx = indexer.lineage.p2i({'descendants': 5})
print(f"{{descendants: 5}}: index {idx}")

# Also accepts dataclass
idx = indexer.lineage.p2i(props)
print(f"Dataclass: index {idx}")

### Combinatorial State Space with multiple properties

In [ ]:
# Two properties: derived allele status (0 or 1) × descendants (0 to 4)
indexer2 = StateIndexer(
    lineage=[
        Property('derived', max_value=1),      # 2 values: 0, 1
        Property('descendants', max_value=4)   # 5 values: 0, 1, 2, 3, 4
    ]
)

print(f"Total states: {indexer2.state_length} (2 × 5 = 10)")
print(f"\nAll states:")
for i in indexer2.lineage:
    props = indexer2.lineage.i2p(i)
    print(f"  Index {i:2d}: derived={props.derived}, descendants={props.descendants}")

## 12. Two-locus ARG model

Her udvider jeg modellen til to loci, hvilket gør det muligt at modellere rekombination.

In [ ]:
nr_samples = 4
indexer = StateIndexer(
    n_descendants=[
        Property('locus1', max_value=nr_samples),
        Property('locus2', max_value=nr_samples)
    ]
)
indexer

In [ ]:
indexer.state_length

In [ ]:
indexer.n_descendants

In [ ]:
print('index', 'locus1', 'locus2', sep='\t')

for i in indexer.n_descendants: 
    props = indexer.n_descendants.index_to_props(i) 
    print(i, props.locus1, props.locus2, sep='\t')

In [ ]:
props = indexer.n_descendants.index_to_props(5) 
props.locus1, props.locus2

In [ ]:
indexer.n_descendants.props_to_index(props)

In [ ]:
indexer.n_descendants.props_to_index(locus1=props.locus1 + 1, 
                                     locus2=props.locus2)

In [ ]:
# zero state vector with appropriate size
initial = [0] * indexer.state_length

# set initial state with all lineages having one descendant at both loci
initial[indexer.props_to_index(locus1=1, locus2=1)] = nr_samples

@with_ipv(initial)
def two_locus_arg(state, indexer=None, N=None, R=None):

    transitions = []
    if state.sum() <= 1: return transitions

    for i in range(indexer.state_length):
        if state[i] == 0: continue
        props_i = indexer.n_descendants.index_to_props(i)

        for j in range(i, indexer.state_length):
            if state[j] == 0: continue
            props_j = indexer.n_descendants.index_to_props(j)
            
            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue 
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            locus1 = props_i.locus1 + props_j.locus1
            locus2 = props_i.locus2 + props_j.locus2
            if locus1 <= nr_samples and locus2 <= nr_samples:
                child[indexer.props_to_index(locus1=locus1, locus2=locus2)] += 1
                transitions.append([child, state[i]*(state[j]-same)/(1+same)/N])

        if state[i] > 0 and props_i.locus1 > 0 and props_i.locus2 > 0:
            child = state.copy()
            child[i] -= 1
            child[indexer.props_to_index(locus1=props_i.locus1, locus2=0)] += 1
            child[indexer.props_to_index(locus1=0, locus2=props_i.locus2)] += 1
            transitions.append([child, R])

    return transitions


graph = Graph(two_locus_arg, 
                     N=1, R=1,
                     indexer=indexer) # passing indexer as argument
graph.plot(max_nodes=200)

In [ ]:
graph.expectation()

In [ ]:
reward_matrix = graph.states().T

In [ ]:
idx_singleton_loc1 = indexer.n_descendants.props_to_index(locus1=1)
idx_singleton_loc1

In [ ]:
reward_matrix = graph.states().T

singleton_rewards_loc1 = reward_matrix[idx_singleton_loc1].sum(axis=0)
singleton_rewards_loc1

In [ ]:
graph.expectation(singleton_rewards_loc1)

In [ ]:
sfs_loc1 = [graph.expectation(reward_matrix[indexer.n_descendants.props_to_index(locus1=i)].sum(axis=0)) for i in range(1, nr_samples+1)]
labels = [f"{i+1}'ton" for i in range(nr_samples)]
sns.barplot(x=labels, y=sfs_loc1, hue=labels) ; 